# Pipeline de profiling taxonomique (SingleM)

## Ce notebook fait quoi ?
Ce notebook automatise un workflow de profiling taxonomique a partir de lectures FASTQ (`*_R1.fastq.gz`) avec **SingleM**, puis construit une matrice d'abondance par echantillon pour exploration.

## Entrees attendues
- Dossier des reads: `data/fastq`
- Convention de nommage: `sample_R1.fastq.gz` (et potentiellement `R2` selon le contexte)
- Base SingleM disponible sur l'infrastructure (`SINGLEM_DB`)

## Etapes executees
1. Configuration des chemins (`DATA_DIR`, `RESULT_DIR`).
2. Detection des echantillons a partir des fichiers FASTQ.
3. Execution de `singlem pipe` puis `singlem summarise` pour chaque echantillon.
4. Agregation des resultats (`taxonomy`, `abundance`) en une table unique.
5. Export de la matrice combinee (`combined_singlem_abundance.tsv`).
6. Filtrage et clustering hierarchique pour visualiser les profils.

## Sorties principales
- Resultats par echantillon dans `results_profiling/singlem/`
- Matrice combinee: `results_profiling/combined_singlem_abundance.tsv`

## Points de vigilance
- Verifier que les modules/commandes SingleM sont disponibles sur votre environnement.
- Adapter les chemins de base de donnees avant execution.
- Les cellules `!module load ...` sont dependantes du cluster/HPC.

In [ ]:
import os
import glob
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist

In [ ]:
# --- CONFIGURATION DES PATHS ---
DATA_DIR = "data/fastq"  # Dossier contenant vos .fastq.gz
RESULT_DIR = "results_profiling"
os.makedirs(RESULT_DIR, exist_ok=True)

In [ ]:
# Chemins des bases de données (à adapter selon votre cluster)
SYLPH_DB = "/shared/bank/gtdb/current/sylph/gtdb-r214.sylphdb"
SINGLEM_DB = "/shared/bank/singlem/current/metapackage" # Si nécessaire

In [ ]:
# Liste des échantillons (on suppose un format sample_R1.fastq.gz)
fastq_r1s = sorted(glob.glob(f"{DATA_DIR}/*_R1.fastq.gz"))
samples = [os.path.basename(f).split('_R1')[0] for f in fastq_r1s]

In [ ]:
singlem_out = f"{RESULT_DIR}/singlem"
os.makedirs(singlem_out, exist_ok=True)

In [ ]:
for r1 in fastq_r1s:
    sample = os.path.basename(r1).split('_R1')[0]
    otu_table = f"{singlem_out}/{sample}.otu.json"
    tax_profile = f"{singlem_out}/{sample}_tax.tsv"
    
    print(f"🚀 Processing {sample} with SingleM...")
    
    # SingleM nécessite souvent deux étapes : pipe (calcul) puis summarise (taxonomie)
    !module load singlem && \
     singlem pipe --sequences {r1} --otu-table {otu_table} --threads 8 && \
     singlem summarise --otu-table {otu_table} --taxonomy-out {tax_profile}

In [ ]:
# Agrégation des résultats SingleM
singlem_tables = []
for s in samples:
    path = f"{singlem_out}/{s}_tax.tsv"
    if os.path.exists(path):
        df = pd.read_csv(path, sep='\t')
        # SingleM fournit 'taxonomy' et 'abundance'
        df = df[['taxonomy', 'abundance']].rename(columns={'abundance': s})
        singlem_tables.append(df.set_index('taxonomy'))

In [ ]:
df_singlem_final = pd.concat(singlem_tables, axis=1).fillna(0)
df_singlem_final.to_csv(f"{RESULT_DIR}/combined_singlem_abundance.tsv", sep='\t')

In [ ]:
# Choix de la table à analyser
data_to_plot = df_sylph_final.copy()

# Filtrage : On ne garde que les taxons qui atteignent au moins 1% quelque part
data_filtered = data_to_plot[data_to_plot.max(axis=1) >= 0.01]

print(f"Nombre de taxons après filtrage (>1%) : {len(data_filtered)}")

In [ ]:
# --- CLUSTERING HIERARCHIQUE ---
# Calcul de la distance (Euclidienne) et du lien (Ward)
Z = linkage(pdist(data_filtered.T), method='ward')

In [ ]:
# Affichage du Clustermap
g = sns.clustermap(data_filtered, 
                   method='ward', 
                   cmap="YlGnBu", 
                   figsize=(12, 10),
                   xticklabels=True, 
                   yticklabels=True,
                   cbar_kws={'label': 'Abondance Relative'})

plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha='right')
plt.suptitle(f"Heatmap de clustering hiérarchique (Top {len(data_filtered)} taxons)", fontsize=16)
plt.show()

In [ ]:
singlem summarise \
     --input-taxonomic-profile {profiles_list} \
     --output-species-by-site-relative-abundance-prefix {matrix_prefix}
     # produces: myprefix-domain.tsv, myprefix-phylum.tsv, ..., myprefix-species.tsv